In [ ]:
from decimal import Decimal
import xarray as xr
import pandas as pd
from nautilus_trader.backtest.engine import BacktestEngine
from nautilus_trader.config import BacktestEngineConfig
from nautilus_trader.config import LoggingConfig
from nautilus_trader.model import TraderId
from nautilus_trader.model.currencies import USD
from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import BarType
from nautilus_trader.model.enums import AccountType
from nautilus_trader.model.enums import OmsType
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.objects import Money
from nautilus_trader.persistence.wranglers import BarDataWrangler
from nautilus_trader.adapters.binance import BINANCE_VENUE
from nautilus_trader.test_kit.providers import TestInstrumentProvider
from nautilus_trader.core.correctness import PyCondition
from nautilus_trader.core.datetime import dt_to_unix_nanos
from nautilus_trader.core.datetime import secs_to_nanos
from nautilus_trader.core.uuid import UUID4
from nautilus_trader.model.currencies import ADA
from nautilus_trader.model.currencies import AUD
from nautilus_trader.model.currencies import BTC
from nautilus_trader.model.currencies import ETH
from nautilus_trader.model.currencies import GBP
from nautilus_trader.model.currencies import USDC
from nautilus_trader.model.currencies import USDT
from nautilus_trader.model.currencies import XRP
from nautilus_trader.model.data import Bar
from nautilus_trader.model.data import QuoteTick
from nautilus_trader.model.data import TradeTick
from nautilus_trader.model.enums import AggressorSide
from nautilus_trader.model.enums import AssetClass
from nautilus_trader.model.enums import OptionKind
from nautilus_trader.model.identifiers import InstrumentId
from nautilus_trader.model.identifiers import Symbol
from nautilus_trader.model.identifiers import TradeId
from nautilus_trader.model.identifiers import Venue
from nautilus_trader.model.instruments import BettingInstrument
from nautilus_trader.model.instruments import BinaryOption
from nautilus_trader.model.instruments import Cfd
from nautilus_trader.model.instruments import CryptoFuture
from nautilus_trader.model.instruments import CryptoPerpetual
from nautilus_trader.model.instruments import CurrencyPair
from nautilus_trader.model.instruments import Equity
from nautilus_trader.model.instruments import FuturesContract
from nautilus_trader.model.instruments import Instrument
from nautilus_trader.model.instruments import OptionContract
from nautilus_trader.model.instruments import SyntheticInstrument
from nautilus_trader.model.instruments.betting import null_handicap
from nautilus_trader.model.objects import Currency
from nautilus_trader.model.objects import Money
from nautilus_trader.model.objects import Price
from nautilus_trader.model.objects import Quantity
from nautilus_trader.model.enums import CurrencyType
from nautilus_trader.persistence.loaders import CSVBarDataLoader
from nautilus_trader.persistence.loaders import CSVTickDataLoader
from nautilus_trader.persistence.loaders import ParquetBarDataLoader
from nautilus_trader.persistence.loaders import ParquetTickDataLoader
# from nautilus_trader.persistence.catalog.types import CatalogWriteMode
from decimal import Decimal

In [ ]:
Currency.from_str("SDD")

In [ ]:
ETH

In [ ]:
BTC

In [ ]:
Currency(code="BTC", precision=8, iso4217=0, name="Bitcoin", currency_type=CurrencyType.CRYPTO)

In [ ]:
btcusdt = CurrencyPair(
    instrument_id=InstrumentId(
        symbol=Symbol("BTCUSDT"),
        venue=Venue("BINANCE"),
    ),
    raw_symbol=Symbol("BTCUSDT"),
    base_currency=BTC,
    quote_currency=USDT,
    price_precision=2,
    size_precision=6,
    price_increment=Price(1e-02, precision=2),
    size_increment=Quantity(1e-06, precision=6),
    lot_size=None,
    max_quantity=Quantity(9000, precision=6),
    min_quantity=Quantity(1e-06, precision=6),
    max_notional=None,
    min_notional=Money(10.00000000, USDT),
    max_price=Price(1000000, precision=2),
    min_price=Price(0.01, precision=2),
    margin_init=Decimal(0),
    margin_maint=Decimal(0),
    maker_fee=Decimal("0.001"),
    taker_fee=Decimal("0.001"),
    ts_event=0,
    ts_init=0,
)

In [ ]:
btcusdt.id

In [ ]:
bt = BarType.from_str(
    f"{btcusdt.id}-1-MINUTE-LAST-EXTERNAL",
)

In [ ]:
bt

In [ ]:
wrangler = BarDataWrangler(bar_type=bt, instrument=btcusdt)

In [1]:
from dataset.spot import SpotKlineDataset
from config import spot_kline_config

In [3]:
ds = SpotKlineDataset(spot_kline_config(symbols=("BTCUSDT",), start_date='2025-01-01'))

In [4]:
bars = None
for b in ds.to_nautilus():
    bars = b
    break

/home/zhrdai/projects/crypto_quant/.venv/lib/python3.12/site-packages/zarr/codecs/vlen_utf8.py:44: UserWarning: The codec `vlen-utf8` is currently not part in the Zarr format 3 specification. It may not be supported by other zarr implementations and may change in the future.
  return cls(**configuration_parsed)


In [ ]:
df = ds.read().get_xarray_dataset()

In [ ]:
df

In [ ]:
for symbol, d in df.groupby('symbol'):
    # print(d.dropna(dim='timestamp').to_dataframe())
    print(symbol.split('USDT')[0])
    break

In [ ]:
# ["timestamp", "open", "high", "low", "close", "volume"]
df = df.reset_index()

In [ ]:
df = df.rename({'Close': 'close', 'Open': 'open', 'High': 'high', 'Low': 'low', 'Volume': 'volume'}, axis=1)

In [ ]:
df = df[["timestamp", "open", "high", "low", "close", "volume"]]

In [ ]:
df = df.dropna()

In [ ]:
df = df.set_index('timestamp')

In [ ]:
bars = wrangler.process(df)

In [ ]:
from pathlib import Path
from nautilus_trader.persistence.catalog import ParquetDataCatalog

In [ ]:
Path.cwd()

In [ ]:
CATALOG_PATH = Path.cwd() / "catalog"

# Create a new catalog instance
catalog = ParquetDataCatalog(CATALOG_PATH, fs_protocol="file")

In [ ]:
catalog.query(btcusdt)

In [ ]:
catalog.write_data(bars)

In [ ]:
catalog.query()